# Example 16.1 / 2: Place Queens With Binary Variables

Companion notebook for **16.1 N-Queens** in the AMPL lab Documentation, section "Place Queens With Binary Variables". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Place Queens With Binary Variables

A second way to model placement puzzles uses a binary variable for every square: `Place[r,c] = 1` when a queen stands on row `r`, column `c`. One queen per row and per column becomes a sum equal to 1, and at most one queen per diagonal becomes a sum at most 1. Sudoku Solver (AMPL lab documentation, section 16.2), Sudoku Generator (AMPL lab documentation, section 16.3) and Nonograms (AMPL lab documentation, section 16.6) use this binary placement style.

The model below includes the binary no-good cut: if a recorded board has its queens on the squares in `FOUND[s]`, then `sum {(r,c) in FOUND[s]} Place[r,c] <= n - 1` forbids placing all $n$ of them again. The run counts the boards for $n=4,5,6$ and reads `Place` with `to_dict()`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param n integer > 0;  # board size and number of queens
set ROWS := 1..n;  # board rows
set COLS := 1..n;  # board columns
var Place {ROWS, COLS} binary;  # 1 if a queen stands on (r,c)
param nfound integer >= 0 default 0;  # solutions recorded so far
set FOUND {1..nfound} within {ROWS, COLS};  # queen squares of each recorded board

subject to OnePerRow {r in ROWS}:  # one queen in each row
    sum {c in COLS} Place[r,c] = 1;
subject to OnePerColumn {c in COLS}:  # one queen in each column
    sum {r in ROWS} Place[r,c] = 1;
subject to DownDiagonal {d in 2..2*n}:  # at most one queen where r + c = d
    sum {r in ROWS, c in COLS: r + c = d} Place[r,c] <= 1;
subject to UpDiagonal {d in 1-n..n-1}:  # at most one queen where r - c = d
    sum {r in ROWS, c in COLS: r - c = d} Place[r,c] <= 1;
subject to Exclude {s in 1..nfound}:  # no-good cut: not all queens of board s again
    sum {(r,c) in FOUND[s]} Place[r,c] <= n - 1;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
sizes = [4, 5, 6]

In [5]:
# run.py
for n in sizes:
    ampl.param["n"] = n
    ampl.param["nfound"] = 0
    while True:
        ampl.solve(solver="highs", verbose=False)
        if ampl.solve_result != "solved":
            break
        place = ampl.var["Place"].to_dict()  # read before nfound changes the model
        k = int(ampl.param["nfound"].value()) + 1
        ampl.param["nfound"] = k
        ampl.set["FOUND"][k] = [rc for rc, v in place.items() if v > 0.5]
    count = int(ampl.param["nfound"].value())
    print(f"n = {n}: {count} solutions, final status {ampl.solve_result}")

n = 4: 2 solutions, final status infeasible
n = 5: 10 solutions, final status infeasible
n = 6: 4 solutions, final status infeasible


### Pure AMPL: data.dat and model.run

The same count for $n=4$ in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param n := 4;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs, solver_msg 0;
repeat {
    solve;
    if solve_result <> "solved" then break;
    let nfound := nfound + 1;
    let FOUND[nfound] := {r in ROWS, c in COLS: Place[r,c] > 0.5};
};
printf "%d solutions, final status %s\n", nfound, solve_result;

2 solutions, final status infeasible
